# NCAIR-DSA — Evaluation Notebook
### Dataset, ASR Accuracy, Clinical Extraction Quality, Error Analysis, Pipeline Experiments

**Purpose:** This is the data-science evaluation layer sitting on top of the built application. The app itself (ASR, N-ATLaS, database, UI) is not being retrained or modified here — this notebook measures how well the existing pipeline performs, using a held-out test set.

**Terminology note (per supervisor feedback):** the audio/text samples used below are a **test/evaluation set**, not a training set. We are not training or fine-tuning any model — N-ATLaS and the NCAIR ASR models are provided, pre-trained models we evaluate the pipeline around, not against.

**What this notebook covers:**
1. Dataset / EDA — what we're testing on, how it's distributed
2. ASR evaluation — WER/CER, using `jiwer`
3. Clinical information extraction evaluation — custom precision/recall-style metrics against ground-truth annotations
4. Error analysis — specific failure categories (negation, misattribution, resolved symptoms, code-switching, ASR-induced errors)
5. Pipeline experiments — does preprocessing (noise reduction, amplification) actually help, or are we assuming it does?
6. Summary / conclusions

**A note on metrics:** per supervisor guidance, we are not required to use a single standardized evaluation metric — some of what's measured here (e.g. Field Capture Accuracy) is a metric we defined ourselves, appropriate to what we're actually trying to verify. This is stated explicitly wherever a custom (non-standard) metric is used.

## 1. Setup

In [ ]:
!pip install -q jiwer pandas matplotlib seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from jiwer import wer, cer
import json

sns.set_style("whitegrid")
print("Setup complete.")

## 2. Dataset / EDA

**Test set structure.** Each row represents one test sample: an audio clip, its ground-truth transcript (what was actually said, written down by a fluent speaker), and ground-truth clinical annotations (what a human would extract as chief complaint, duration, severity, history).

Replace `test_manifest.csv` with your actual test set — this notebook expects the following columns:

| Column | Description |
|---|---|
| `sample_id` | Unique ID per test sample |
| `audio_path` | Path to the audio file |
| `language` | Hausa / Igbo / Yoruba |
| `audio_duration_sec` | Length of the clip in seconds |
| `ground_truth_transcript` | What was actually said (human-transcribed, original language) |
| `gt_chief_complaint` | Human-annotated chief complaint (English) |
| `gt_duration` | Human-annotated duration |
| `gt_severity` | Human-annotated severity (mild/moderate/severe) |
| `gt_history` | Human-annotated relevant history |
| `test_category` | e.g. "standard", "negation", "family_history", "code_switching", "resolved_symptom" — used for error analysis in Section 4 |

**How many samples do you realistically need?** For a coursework-scale evaluation, aim for at least 5-10 samples per language (15-30 total minimum) to say anything meaningful about per-language ASR accuracy. More is better, but be honest in your writeup about sample size — a small test set is fine to report on, as long as you don't overclaim statistical significance from it.

In [ ]:
# Load your test manifest — replace with the real file once your team has
# collected and annotated test samples
try:
    df = pd.read_csv("test_manifest.csv")
    print(f"Loaded {len(df)} test samples")
except FileNotFoundError:
    print("test_manifest.csv not found — using a small illustrative example instead.")
    print("Replace this with your real annotated test set before drawing conclusions.")
    df = pd.DataFrame({
        "sample_id": ["S001", "S002", "S003"],
        "audio_path": ["audio/s001.wav", "audio/s002.wav", "audio/s003.wav"],
        "language": ["Hausa", "Yoruba", "Igbo"],
        "audio_duration_sec": [8.2, 12.5, 6.1],
        "ground_truth_transcript": ["...", "...", "..."],
        "gt_chief_complaint": ["Abdominal pain", "Headache", "Fever"],
        "gt_duration": ["2 days", "1 week", "3 days"],
        "gt_severity": ["moderate", "mild", "severe"],
        "gt_history": ["None", "None", "None"],
        "test_category": ["standard", "standard", "standard"],
    })

df.head()

In [ ]:
# Basic distribution: samples per language
print("Samples per language:")
print(df["language"].value_counts())
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["language"].value_counts().plot(kind="bar", ax=axes[0], color=["#2C5F8A", "#4A90A4", "#7BAFB0"])
axes[0].set_title("Test Samples per Language")
axes[0].set_ylabel("Count")

df["audio_duration_sec"].hist(bins=10, ax=axes[1], color="#2C5F8A")
axes[1].set_title("Audio Duration Distribution")
axes[1].set_xlabel("Duration (seconds)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Test category breakdown — this matters for the error analysis section later
print("Samples per test category:")
print(df["test_category"].value_counts())

# Flag any missing/problematic data
print("\nMissing values per column:")
print(df.isnull().sum())

## 3. ASR Evaluation — Word Error Rate (WER) / Character Error Rate (CER)

These are standard, established ASR metrics — not something we're coining ourselves. Lower is better; 0.0 = perfect match.

**What this measures:** how accurately the NCAIR ASR models transcribe speech in each language, compared to what a human says was actually spoken.

In [ ]:
# Run this after transcribing each audio_path with your actual transcribe_audio() function.
# For now this expects a "system_transcript" column — replace the placeholder
# logic below with real calls to your ASR pipeline.

# from asr.transcribe import transcribe_audio
# df["system_transcript"] = df.apply(
#     lambda row: transcribe_audio(row["audio_path"], row["language"]), axis=1
# )

# Placeholder for demonstration — replace with real transcription output
if "system_transcript" not in df.columns:
    df["system_transcript"] = df["ground_truth_transcript"]  # REPLACE with real ASR output

df["wer"] = df.apply(lambda row: wer(row["ground_truth_transcript"], row["system_transcript"]), axis=1)
df["cer"] = df.apply(lambda row: cer(row["ground_truth_transcript"], row["system_transcript"]), axis=1)

print("Overall WER:", df["wer"].mean())
print("Overall CER:", df["cer"].mean())

df[["sample_id", "language", "wer", "cer"]]

In [ ]:
# Per-language breakdown — this is the number your supervisor will likely
# ask about directly, since accuracy is not expected to be uniform across
# Hausa/Igbo/Yoruba
wer_by_language = df.groupby("language")[["wer", "cer"]].mean().sort_values("wer")
print(wer_by_language)

wer_by_language["wer"].plot(kind="bar", figsize=(7, 4), color="#2C5F8A", title="Average WER by Language")
plt.ylabel("Word Error Rate")
plt.tight_layout()
plt.show()

## 4. Clinical Information Extraction Evaluation (Custom Metrics)

**Important framing:** chief_complaint/duration/severity/history are free-text fields, not fixed categories — so standard precision/recall (which assumes discrete labels) doesn't apply cleanly. Per supervisor guidance, we define our own metrics here, chosen to measure what actually matters for this task.

### Custom Metric 1 — Field Capture Accuracy
Did the system correctly identify whether a field was present or should be "Not mentioned"? This is a simple, honest binary check: for each field, does the system's presence/absence match the ground truth's presence/absence.

### Custom Metric 2 — Symptom Keyword Recall
Of the clinically meaningful terms a human annotator identified in the ground truth, what fraction appear (in some form) in the system's chief_complaint/history output? This is a recall-style measure adapted for free text, not exact-match precision/recall.

### Custom Metric 3 — Severity Agreement
Does the system's severity classification (mild/moderate/severe) match the human annotator's judgment? This one — unlike the others — genuinely is a discrete category, so plain accuracy applies directly here, no adaptation needed.

In [ ]:
def field_capture_accuracy(system_value: str, ground_truth_value: str) -> bool:
    """
    Custom metric: does the system correctly capture whether a field has
    real content, vs. correctly leaving it as "Not mentioned"?
    This checks PRESENCE match, not exact wording match.
    """
    system_has_content = system_value.strip().lower() not in ["", "not mentioned", "none"]
    gt_has_content = ground_truth_value.strip().lower() not in ["", "not mentioned", "none"]
    return system_has_content == gt_has_content


def symptom_keyword_recall(system_text: str, ground_truth_keywords: list) -> float:
    """
    Custom metric: of the ground-truth-identified keywords, what fraction
    appear (as a substring, case-insensitive) in the system's output text?
    """
    if not ground_truth_keywords:
        return None  # not applicable — no keywords to check against
    system_lower = system_text.lower()
    found = sum(1 for kw in ground_truth_keywords if kw.lower() in system_lower)
    return found / len(ground_truth_keywords)


# Example usage — wire this up once you have real system output per field
# (from structure_note() run on each test sample)
example_system_output = {
    "chief_complaint": "Abdominal pain with vomiting",
    "duration": "Since yesterday",
    "severity": "severe",
    "history": "Not mentioned",
}
example_ground_truth = {
    "chief_complaint": "Abdominal pain, vomiting",
    "duration": "1 day",
    "severity": "severe",
    "history": "Not mentioned",
}

for field in ["chief_complaint", "duration", "severity", "history"]:
    match = field_capture_accuracy(example_system_output[field], example_ground_truth[field])
    print(f"{field}: capture match = {match}")

recall = symptom_keyword_recall(
    example_system_output["chief_complaint"],
    ["abdominal pain", "vomiting"]
)
print(f"\nSymptom keyword recall: {recall:.0%}")

In [ ]:
# Run field capture accuracy across the whole test set
# (once you have real structure_note() output stored per sample — replace
# the placeholder columns below with real system output)

# from nlp.structure_note import structure_note
# results = df["ground_truth_transcript"].apply(structure_note)
# df["sys_chief_complaint"] = results.apply(lambda r: r["chief_complaint"])
# df["sys_duration"] = results.apply(lambda r: r["duration"])
# df["sys_severity"] = results.apply(lambda r: r["severity"])
# df["sys_history"] = results.apply(lambda r: r["history"])

# Placeholder — replace with real system output
for field in ["chief_complaint", "duration", "severity", "history"]:
    if f"sys_{field}" not in df.columns:
        df[f"sys_{field}"] = df[f"gt_{field}"]  # REPLACE with real structure_note() output

for field in ["chief_complaint", "duration", "severity", "history"]:
    df[f"{field}_capture_match"] = df.apply(
        lambda row: field_capture_accuracy(row[f"sys_{field}"], row[f"gt_{field}"]), axis=1
    )

capture_accuracy_summary = df[[f"{f}_capture_match" for f in
                                ["chief_complaint", "duration", "severity", "history"]]].mean()
print("Field Capture Accuracy (custom metric):")
print(capture_accuracy_summary)

In [ ]:
# Severity agreement — this one IS a standard accuracy metric, since
# severity is a genuine discrete category (mild/moderate/severe)
severity_accuracy = (df["sys_severity"].str.lower() == df["gt_severity"].str.lower()).mean()
print(f"Severity classification accuracy: {severity_accuracy:.1%}")

## 5. Error Analysis

The point of this section isn't to compute a summary number — it's to look directly at specific failure categories your supervisor named explicitly:

- **Negation**: "I don't have fever" should NOT be captured as a fever symptom
- **Misattribution**: "My father has diabetes" should NOT become the patient's own history
- **Resolved/past symptoms**: something the patient says has already gone away shouldn't be treated as a current complaint
- **Code-switching**: mixed-language input, where translation/structuring may break down
- **ASR-induced errors**: cases where a transcription mistake causes N-ATLaS to structure the wrong information downstream

Construct test cases deliberately for each category — don't just wait to stumble onto them in random samples.

In [ ]:
# Deliberately constructed test cases per failure category.
# Fill in real audio/transcripts once available; text-only versions shown
# here so the LLM-level failure mode can be checked independently of ASR.

error_analysis_cases = [
    {
        "category": "negation",
        "transcript": "I don't have any fever, just a mild headache",
        "should_not_contain": ["fever"],
        "should_contain": ["headache"],
    },
    {
        "category": "family_history_misattribution",
        "transcript": "My father has diabetes, but I just have a stomach ache since this morning",
        "should_not_contain": ["diabetes"],  # should not appear as the PATIENT'S complaint
        "should_contain": ["stomach"],
    },
    {
        "category": "resolved_symptom",
        "transcript": "I had a fever last week but it went away, now I just have a cough",
        "should_not_contain": [],  # fever may appear in history, but should not be chief_complaint
        "should_contain": ["cough"],
    },
]

# from nlp.structure_note import structure_note
# for case in error_analysis_cases:
#     result = structure_note(case["transcript"])
#     print(f"--- {case[\'category\'].upper()} ---")
#     print(f"Transcript: {case[\'transcript\']}")
#     print(f"System chief_complaint: {result[\'chief_complaint\']}")
#     print(f"System history: {result[\'history\']}")
#     for term in case["should_not_contain"]:
#         flag = "FAIL" if term.lower() in result["chief_complaint"].lower() else "PASS"
#         print(f"  [{flag}] Should NOT contain \'{term}\'")
#     print()

print("Wire this up to structure_note() once running in Colab with the model loaded.")
print("Each case above is a deliberately constructed failure-mode probe, not a random sample.")

In [ ]:
# Manual error log template — fill this in as you run real test cases.
# This becomes your qualitative error analysis section for the writeup.

error_log = pd.DataFrame({
    "sample_id": [],
    "category": [],
    "transcript": [],
    "system_output": [],
    "expected_behavior": [],
    "pass_fail": [],
    "notes": [],
})

print("Error log template ready — populate as you review real test outputs.")
error_log

## 6. Pipeline Experiments — Does Preprocessing Actually Help?

This directly tests an assumption rather than just stating it: does the noise reduction / amplification preprocessing in `asr/transcribe.py` actually improve WER, or are we just assuming it does?

**Method:** run the same audio samples through ASR twice — once completely raw, once through the full preprocessing pipeline — and compare WER.

In [ ]:
# from asr.transcribe import transcribe_audio, preprocess_audio, reduce_noise, amplify_audio
# import soundfile as sf

# results = []
# for _, row in df.iterrows():
#     # Raw (no preprocessing) — bypass the cleanup steps
#     raw_result = raw_asr_pipeline(row["audio_path"], row["language"])  # define a raw-only variant
#     raw_wer = wer(row["ground_truth_transcript"], raw_result)
#
#     # Full pipeline (with preprocessing)
#     processed_result = transcribe_audio(row["audio_path"], row["language"])
#     processed_wer = wer(row["ground_truth_transcript"], processed_result)
#
#     results.append({
#         "sample_id": row["sample_id"],
#         "raw_wer": raw_wer,
#         "processed_wer": processed_wer,
#         "improvement": raw_wer - processed_wer,
#     })

# comparison_df = pd.DataFrame(results)
# print(comparison_df)
# print(f"\nMean WER without preprocessing: {comparison_df[\'raw_wer\'].mean():.3f}")
# print(f"Mean WER with preprocessing:    {comparison_df[\'processed_wer\'].mean():.3f}")

print("Wire this up once running in Colab. This produces the actual evidence")
print("for whether preprocessing helps — don't state it helps without this comparison.")

## 7. Summary / Conclusions

Fill this in once the sections above are populated with real data. Suggested structure for your writeup:

- **ASR performance**: overall and per-language WER/CER — which language performs best/worst, and does this match what was expected from the NCAIR/N-ATLaS model card's published scores?
- **Clinical extraction quality**: Field Capture Accuracy and Severity Agreement — where does the system do well, where does it struggle?
- **Error analysis findings**: which failure categories (negation, misattribution, resolved symptoms, code-switching) were actual problems vs. handled correctly?
- **Preprocessing impact**: did noise reduction/amplification measurably help, hurt, or make no difference?
- **Honest limitations**: sample size, what wasn't tested, what would need a larger evaluation to say more confidently

**Terminology reminder for presentation day:** refer to this as an *evaluation* or *test set*, never a "training set" — nothing here retrains any model. Reference "custom metrics we defined" explicitly when presenting Field Capture Accuracy / Symptom Keyword Recall, since these aren't standard, established metrics — say so plainly rather than presenting them as if they were.